# Split MNIST Continual Learning Comparison

This notebook demonstrates a comparison of different continual learning methods on the Split MNIST benchmark.

The task sequence consists of two tasks:
- **Task 1**: MNIST classes 0–4
- **Task 2**: MNIST classes 5–9

Methods compared:
- **Baseline**: Naive fine-tuning (demonstrates catastrophic forgetting)
- **EWC**: Elastic Weight Consolidation
- **Replay**: Experience Replay
- **LwF**: Learning without Forgetting

In [ ]:
import os
import sys

# Ensure the repository root is on the Python path
repo_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

import numpy as np
import torch
import matplotlib.pyplot as plt

from src.data.data_loader import get_task_sequence
from src.models.model_factory import get_model
from src.methods.baseline import BaselineLearner
from src.methods.ewc import EWCLearner
from src.methods.replay import ExperienceReplayLearner
from src.methods.lwf import LwFLearner
from src.utils.metrics import evaluate_performance, compute_forgetting
from src.utils.visualization import plot_performance, plot_forgetting

print('Imports successful')

In [ ]:
# Configuration
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
BATCH_SIZE = 64
EPOCHS = 3
LEARNING_RATE = 0.001
SEED = 42

# Set random seed for reproducibility
torch.manual_seed(SEED)
np.random.seed(SEED)

print(f'Using device: {DEVICE}')

In [ ]:
# Define the Split MNIST task sequence
task_sequence = [
    {'name': 'mnist_0_4', 'dataset': 'mnist', 'classes': [0, 1, 2, 3, 4]},
    {'name': 'mnist_5_9', 'dataset': 'mnist', 'classes': [5, 6, 7, 8, 9]},
]

# Load data
task_data = get_task_sequence(task_sequence, BATCH_SIZE)

# Determine input shape and number of classes
input_shape = task_data[0]['train_loader'].dataset[0][0].shape
all_classes = set()
for task in task_sequence:
    all_classes.update(task['classes'])
num_classes = len(all_classes)

print(f'Input shape: {input_shape}')
print(f'Number of classes: {num_classes}')
print(f'Tasks: {[t["name"] for t in task_sequence]}')

In [ ]:
def run_experiment(learner, task_data, task_sequence):
    """Train a learner sequentially on all tasks and record performance."""
    n_tasks = len(task_sequence)
    performance_matrix = np.zeros((n_tasks, n_tasks))

    for task_id, task_dict in enumerate(task_data):
        train_loader = task_dict['train_loader']
        val_loader = task_dict['val_loader']
        print(f'  Training on task {task_id + 1}/{n_tasks}: {task_sequence[task_id]["name"]}')

        learner.train(
            train_loader=train_loader,
            val_loader=val_loader,
            task_id=task_id,
            epochs=EPOCHS,
            eval_freq=1,
        )

        for eval_task_id in range(task_id + 1):
            eval_loader = task_data[eval_task_id]['test_loader']
            accuracy = learner.evaluate(eval_loader, eval_task_id)
            performance_matrix[task_id, eval_task_id] = accuracy
            print(f'    Accuracy on task {eval_task_id + 1}: {accuracy:.2f}%')

    return performance_matrix

In [ ]:
results = {}

# --- Baseline ---
print('=== Baseline (Fine-tuning) ===')
model = get_model('simple_cnn', input_shape, num_classes).to(DEVICE)
learner = BaselineLearner(model=model, device=DEVICE, learning_rate=LEARNING_RATE)
results['baseline'] = run_experiment(learner, task_data, task_sequence)

print()

# --- EWC ---
print('=== EWC ===')
model = get_model('simple_cnn', input_shape, num_classes).to(DEVICE)
learner = EWCLearner(model=model, device=DEVICE, learning_rate=LEARNING_RATE,
                     lambda_ewc=5000, fisher_sample_size=200)
results['ewc'] = run_experiment(learner, task_data, task_sequence)

print()

# --- Experience Replay ---
print('=== Experience Replay ===')
model = get_model('simple_cnn', input_shape, num_classes).to(DEVICE)
learner = ExperienceReplayLearner(model=model, device=DEVICE, learning_rate=LEARNING_RATE,
                                  buffer_size=500, replay_batch_size=32)
results['replay'] = run_experiment(learner, task_data, task_sequence)

print()

# --- LwF ---
print('=== LwF ===')
model = get_model('simple_cnn', input_shape, num_classes).to(DEVICE)
learner = LwFLearner(model=model, device=DEVICE, learning_rate=LEARNING_RATE,
                     temperature=2.0, alpha=1.0)
results['lwf'] = run_experiment(learner, task_data, task_sequence)

In [ ]:
task_names = [t['name'] for t in task_sequence]

# Summary table
print('\n=== Final Accuracy on Each Task After All Training ===')
print(f'{"Method":<12}', end='')
for name in task_names:
    print(f'{name:>16}', end='')
print(f'{"Average":>12}')

for method, perf in results.items():
    final_row = perf[-1, :len(task_names)]
    print(f'{method:<12}', end='')
    for acc in final_row:
        print(f'{acc:>15.2f}%', end='')
    print(f'{np.mean(final_row):>11.2f}%')

In [ ]:
# Plot performance matrices
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
method_names = list(results.keys())

for ax, (method, perf) in zip(axes.flatten(), results.items()):
    im = ax.imshow(perf, vmin=0, vmax=100, cmap='Blues', aspect='auto')
    ax.set_title(method.upper())
    ax.set_xlabel('Task evaluated on')
    ax.set_ylabel('After training task')
    ax.set_xticks(range(len(task_names)))
    ax.set_yticks(range(len(task_names)))
    ax.set_xticklabels(task_names, rotation=30, ha='right')
    ax.set_yticklabels(task_names)
    for i in range(len(task_names)):
        for j in range(len(task_names)):
            if perf[i, j] > 0:
                ax.text(j, i, f'{perf[i, j]:.1f}', ha='center', va='center', fontsize=9)
    plt.colorbar(im, ax=ax)

plt.suptitle('Accuracy (%) — Split MNIST Comparison', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Average final accuracy comparison bar chart
avg_accuracies = {
    method: np.mean(perf[-1, :len(task_names)])
    for method, perf in results.items()
}

plt.figure(figsize=(8, 5))
bars = plt.bar(avg_accuracies.keys(), avg_accuracies.values(),
               color=['#4c72b0', '#dd8452', '#55a868', '#c44e52'])
for bar, val in zip(bars, avg_accuracies.values()):
    plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
             f'{val:.1f}%', ha='center', va='bottom')
plt.ylim(0, 110)
plt.ylabel('Average Accuracy (%)')
plt.title('Average Final Accuracy — Split MNIST')
plt.tight_layout()
plt.show()